In [2]:
# ============================================================
# SevaSetu - Urgency Classification using BiLSTM
# ============================================================
#
# INPUT:
#     complaint_text
#
# TARGET:
#     urgency
#
# CLASSES:
#     low
#     medium
#     high
#
# MODEL:
#     TextVectorization
#          ↓
#     Embedding
#          ↓
#     Bidirectional LSTM
#          ↓
#     Dense
#          ↓
#     Softmax
#
# ============================================================


import re
import pickle

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.utils.class_weight import compute_class_weight

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

from tensorflow.keras import Sequential

from tensorflow.keras.layers import (
    TextVectorization,
    Embedding,
    Bidirectional,
    LSTM,
    Dense,
    Dropout,
    BatchNormalization
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATASET_PATH = (
    "sevasetu_grievance_dataset_100k_v1.csv"
)

MODEL_PATH = (
    "sevasetu_urgency_bilstm.keras"
)

LABEL_ENCODER_PATH = (
    "urgency_label_encoder.pkl"
)

VOCAB_PATH = (
    "urgency_vocabulary.pkl"
)


MAX_VOCAB_SIZE = 30000

MAX_SEQUENCE_LENGTH = 150

EMBEDDING_DIM = 128

LSTM_UNITS = 128

BATCH_SIZE = 128

EPOCHS = 15

LEARNING_RATE = 0.001

RANDOM_STATE = 42


# ============================================================
# 2. LOAD DATASET
# ============================================================

print("\n")
print("=" * 60)
print("LOADING DATASET")
print("=" * 60)

df = pd.read_csv(
    DATASET_PATH
)

print(
    "Dataset shape:",
    df.shape
)

print("\nColumns:")

print(
    df.columns.tolist()
)


# ============================================================
# 3. CHECK REQUIRED COLUMNS
# ============================================================

if "complaint_text" not in df.columns:

    raise ValueError(
        "complaint_text column not found!"
    )


if "urgency" not in df.columns:

    raise ValueError(
        "urgency column not found!"
    )


# ============================================================
# 4. SELECT REQUIRED COLUMNS
# ============================================================

data = df[
    [
        "complaint_text",
        "urgency"
    ]
].copy()


# ============================================================
# 5. REMOVE NULL VALUES
# ============================================================

print("\nMissing values:")

print(
    data.isnull().sum()
)


data.dropna(
    subset=[
        "complaint_text",
        "urgency"
    ],
    inplace=True
)


# ============================================================
# 6. CLEAN TEXT
# ============================================================

def clean_text(text):

    text = str(text)

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


data["complaint_text"] = (
    data["complaint_text"]
    .apply(clean_text)
)


# ============================================================
# 7. NORMALIZE URGENCY
# ============================================================

data["urgency"] = (
    data["urgency"]
    .astype(str)
    .str.lower()
    .str.strip()
)


# ============================================================
# 8. KEEP ONLY VALID CLASSES
# ============================================================

valid_classes = [
    "low",
    "medium",
    "high"
]


data = data[
    data["urgency"].isin(
        valid_classes
    )
].copy()


# ============================================================
# 9. REMOVE EXACT DUPLICATES
# ============================================================

before = len(data)


data.drop_duplicates(
    subset=[
        "complaint_text",
        "urgency"
    ],
    inplace=True
)


after = len(data)


print(
    "\nDuplicates removed:",
    before - after
)


print(
    "Remaining rows:",
    after
)


# ============================================================
# 10. SHOW CLASS DISTRIBUTION
# ============================================================

print("\n")
print("=" * 60)
print("URGENCY DISTRIBUTION")
print("=" * 60)


print(
    data["urgency"]
    .value_counts()
)


print("\nPercentage:")


print(
    (
        data["urgency"]
        .value_counts(
            normalize=True
        )
        * 100
    ).round(2)
)


# ============================================================
# 11. LABEL ENCODING
# ============================================================

label_encoder = LabelEncoder()


y = label_encoder.fit_transform(
    data["urgency"]
)


print("\nClasses:")


for i, label in enumerate(
    label_encoder.classes_
):

    print(
        i,
        "->",
        label
    )


# ============================================================
# 12. INPUT
# ============================================================

X = data[
    "complaint_text"
].values


# ============================================================
# 13. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

X_train, X_temp, y_train, y_temp = (
    train_test_split(

        X,

        y,

        test_size=0.20,

        random_state=RANDOM_STATE,

        stratify=y
    )
)


X_val, X_test, y_val, y_test = (
    train_test_split(

        X_temp,

        y_temp,

        test_size=0.50,

        random_state=RANDOM_STATE,

        stratify=y_temp
    )
)


print("\n")
print("=" * 60)
print("DATA SPLIT")
print("=" * 60)


print(
    "Training:",
    len(X_train)
)


print(
    "Validation:",
    len(X_val)
)


print(
    "Testing:",
    len(X_test)
)


# ============================================================
# 14. COMPUTE CLASS WEIGHTS
# ============================================================

classes = np.unique(
    y_train
)


class_weights_array = (
    compute_class_weight(

        class_weight="balanced",

        classes=classes,

        y=y_train
    )
)


class_weights = {

    int(cls): float(weight)

    for cls, weight in zip(
        classes,
        class_weights_array
    )
}


print("\n")
print("=" * 60)
print("CLASS WEIGHTS")
print("=" * 60)


for cls, weight in class_weights.items():

    print(
        label_encoder.inverse_transform(
            [cls]
        )[0],
        "->",
        weight
    )


# ============================================================
# 15. TEXT VECTORIZATION
# ============================================================

print("\n")
print("=" * 60)
print("CREATING TEXT VECTORIZER")
print("=" * 60)


vectorizer = TextVectorization(

    max_tokens=MAX_VOCAB_SIZE,

    output_mode="int",

    output_sequence_length=MAX_SEQUENCE_LENGTH,

    standardize="lower_and_strip_punctuation"
)


# IMPORTANT:
# Learn vocabulary only from training data

vectorizer.adapt(
    X_train
)


vocab_size = len(
    vectorizer.get_vocabulary()
)


print(
    "Vocabulary size:",
    vocab_size
)


# ============================================================
# 16. CREATE TF.DATA DATASETS
# ============================================================

train_dataset = (
    tf.data.Dataset
    .from_tensor_slices(
        (X_train, y_train)
    )
    .shuffle(
        10000
    )
    .batch(
        BATCH_SIZE
    )
    .prefetch(
        tf.data.AUTOTUNE
    )
)


val_dataset = (
    tf.data.Dataset
    .from_tensor_slices(
        (X_val, y_val)
    )
    .batch(
        BATCH_SIZE
    )
    .prefetch(
        tf.data.AUTOTUNE
    )
)


test_dataset = (
    tf.data.Dataset
    .from_tensor_slices(
        (X_test, y_test)
    )
    .batch(
        BATCH_SIZE
    )
    .prefetch(
        tf.data.AUTOTUNE
    )
)


# ============================================================
# 17. BUILD BiLSTM MODEL
# ============================================================

print("\n")
print("=" * 60)
print("BUILDING BiLSTM MODEL")
print("=" * 60)


model = Sequential([

    # --------------------------------------------------------
    # Convert raw text -> integer token sequence
    # --------------------------------------------------------

    vectorizer,


    # --------------------------------------------------------
    # Word Embedding
    # --------------------------------------------------------

    Embedding(

        input_dim=vocab_size,

        output_dim=EMBEDDING_DIM,

        mask_zero=True
    ),


    # --------------------------------------------------------
    # Bidirectional LSTM
    # --------------------------------------------------------

    Bidirectional(

        LSTM(

            LSTM_UNITS,

            return_sequences=False
        )
    ),


    # --------------------------------------------------------
    # Regularization
    # --------------------------------------------------------

    Dropout(
        0.40
    ),


    # --------------------------------------------------------
    # Dense Layer
    # --------------------------------------------------------

    Dense(

        128,

        activation="relu"
    ),


    BatchNormalization(),


    Dropout(
        0.30
    ),


    # --------------------------------------------------------
    # Output Layer
    # --------------------------------------------------------

    Dense(

        len(
            label_encoder.classes_
        ),

        activation="softmax"
    )
])


# ============================================================
# 18. COMPILE MODEL
# ============================================================

model.compile(

    optimizer=tf.keras.optimizers.Adam(

        learning_rate=LEARNING_RATE
    ),

    loss="sparse_categorical_crossentropy",

    metrics=[
        "accuracy"
    ]
)


# ============================================================
# 19. MODEL SUMMARY
# ============================================================

model.summary()


# ============================================================
# 20. CALLBACKS
# ============================================================

callbacks = [

    # --------------------------------------------------------
    # Save best model
    # --------------------------------------------------------

    ModelCheckpoint(

        MODEL_PATH,

        monitor="val_loss",

        save_best_only=True,

        verbose=1
    ),


    # --------------------------------------------------------
    # Stop if validation stops improving
    # --------------------------------------------------------

    EarlyStopping(

        monitor="val_loss",

        patience=3,

        restore_best_weights=True,

        verbose=1
    ),


    # --------------------------------------------------------
    # Reduce learning rate when learning slows
    # --------------------------------------------------------

    ReduceLROnPlateau(

        monitor="val_loss",

        factor=0.5,

        patience=1,

        min_lr=1e-6,

        verbose=1
    )
]


# ============================================================
# 21. TRAIN MODEL
# ============================================================

print("\n")
print("=" * 60)
print("STARTING TRAINING")
print("=" * 60)


history = model.fit(

    train_dataset,

    validation_data=val_dataset,

    epochs=EPOCHS,

    class_weight=class_weights,

    callbacks=callbacks
)


# ============================================================
# 22. LOAD BEST MODEL
# ============================================================

print("\n")
print("=" * 60)
print("LOADING BEST MODEL")
print("=" * 60)


model = tf.keras.models.load_model(
    MODEL_PATH
)


# ============================================================
# 23. TEST ACCURACY
# ============================================================

print("\n")
print("=" * 60)
print("TESTING MODEL")
print("=" * 60)


test_loss, test_accuracy = (
    model.evaluate(

        test_dataset,

        verbose=1
    )
)


print("\nTest Loss:")

print(
    test_loss
)


print("\nTest Accuracy:")

print(
    test_accuracy
)


print(
    f"\nTest Accuracy: "
    f"{test_accuracy * 100:.2f}%"
)


# ============================================================
# 24. GENERATE TEST PREDICTIONS
# ============================================================

print("\nGenerating predictions...")


y_probability = model.predict(

    test_dataset,

    verbose=1
)


y_pred = np.argmax(

    y_probability,

    axis=1
)


# ============================================================
# 25. ACCURACY
# ============================================================

accuracy = accuracy_score(

    y_test,

    y_pred
)


print("\n")
print("=" * 60)
print("FINAL ACCURACY")
print("=" * 60)


print(
    f"{accuracy * 100:.2f}%"
)


# ============================================================
# 26. MACRO F1
# ============================================================

macro_f1 = f1_score(

    y_test,

    y_pred,

    average="macro"
)


weighted_f1 = f1_score(

    y_test,

    y_pred,

    average="weighted"
)


print(
    "\nMacro F1:",
    f"{macro_f1:.4f}"
)


print(
    "Weighted F1:",
    f"{weighted_f1:.4f}"
)


# ============================================================
# 27. CLASSIFICATION REPORT
# ============================================================

print("\n")
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)


print(

    classification_report(

        y_test,

        y_pred,

        target_names=
        label_encoder.classes_,

        digits=4
    )
)


# ============================================================
# 28. CONFUSION MATRIX
# ============================================================

print("\n")
print("=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)


cm = confusion_matrix(

    y_test,

    y_pred
)


cm_df = pd.DataFrame(

    cm,

    index=[
        f"Actual {x}"
        for x in label_encoder.classes_
    ],

    columns=[
        f"Predicted {x}"
        for x in label_encoder.classes_
    ]
)


print(
    cm_df
)


# ============================================================
# 29. SAVE LABEL ENCODER
# ============================================================

with open(

    LABEL_ENCODER_PATH,

    "wb"

) as f:

    pickle.dump(

        label_encoder,

        f
    )


# ============================================================
# 30. SAVE VOCABULARY
# ============================================================

vocabulary = (
    vectorizer
    .get_vocabulary()
)


with open(

    VOCAB_PATH,

    "wb"

) as f:

    pickle.dump(

        vocabulary,

        f
    )


# ============================================================
# 31. FINAL FILE INFORMATION
# ============================================================

print("\n")
print("=" * 60)
print("FILES SAVED")
print("=" * 60)


print(
    "Model:",
    MODEL_PATH
)


print(
    "Label Encoder:",
    LABEL_ENCODER_PATH
)


print(
    "Vocabulary:",
    VOCAB_PATH
)


# ============================================================
# 32. PREDICTION FUNCTION
# ============================================================

def predict_urgency(
    complaint
):

    complaint = str(
        complaint
    )


    # --------------------------------------------------------
    # Predict probabilities
    # --------------------------------------------------------

    probabilities = model.predict(

        np.array([
            complaint
        ]),

        verbose=0
    )[0]


    # --------------------------------------------------------
    # Find best class
    # --------------------------------------------------------

    predicted_index = int(

        np.argmax(
            probabilities
        )
    )


    # --------------------------------------------------------
    # Convert index -> label
    # --------------------------------------------------------

    urgency = (
        label_encoder
        .inverse_transform(
            [predicted_index]
        )[0]
    )


    # --------------------------------------------------------
    # Confidence
    # --------------------------------------------------------

    confidence = float(

        probabilities[
            predicted_index
        ]
    )


    # --------------------------------------------------------
    # All probabilities
    # --------------------------------------------------------

    probability_dict = {

        label_encoder
        .inverse_transform([i])[0]:

        float(
            probabilities[i]
        )

        for i in range(
            len(
                probabilities
            )
        )
    }


    return {

        "urgency":
            urgency,

        "confidence":
            confidence,

        "probabilities":
            probability_dict
    }


# ============================================================
# 33. TEST COMPLETELY NEW COMPLAINTS
# ============================================================

print("\n")
print("=" * 60)
print("UNSEEN COMPLAINT TEST")
print("=" * 60)


new_complaints = [

    # HIGH
    """
    An exposed live electrical wire has fallen onto
    the main road and children are walking nearby.
    Immediate action is required.
    """,

    # LOW
    """
    Please provide information about the documents
    required to apply for a government certificate.
    """,

    # MEDIUM
    """
    The water pipeline near my house has been leaking
    for several days and needs to be repaired.
    """,

    # HIGH
    """
    A fire has started inside a government building
    and people are trapped inside. Emergency help
    is urgently required.
    """,

    # MEDIUM
    """
    I submitted my application several weeks ago
    but it has not been processed yet.
    """,

    # HIGH
    """
    There is a dangerous pothole near a school
    entrance and children are at risk.
    """,

    # LOW
    """
    I want information about the process for updating
    my address in a government record.
    """,

    # Hinglish HIGH
    """
    Hamare area mein transformer se sparks aa rahe hain
    aur exposed wire road par pada hai. Bachchon ke liye
    bahut dangerous hai.
    """
]


for complaint in new_complaints:

    result = predict_urgency(
        complaint.strip()
    )


    print("\nComplaint:")
    print(
        complaint.strip()
    )


    print(
        "\nPredicted Urgency:",
        result["urgency"].upper()
    )


    print(
        "Confidence:",
        f"{result['confidence'] * 100:.2f}%"
    )


    print(
        "\nProbabilities:"
    )


    for label, probability in (
        result["probabilities"].items()
    ):

        print(

            f"  {label:<8}: "
            f"{probability * 100:.2f}%"
        )


    print(
        "-" * 60
    )



LOADING DATASET
Dataset shape: (100000, 8)

Columns:
['complaint_id', 'sector', 'complaint_text', 'location', 'complaint_type', 'phone_number', 'department', 'urgency']

Missing values:
complaint_text    0
urgency           0
dtype: int64

Duplicates removed: 2518
Remaining rows: 97482


URGENCY DISTRIBUTION
urgency
medium    52116
high      36305
low        9061
Name: count, dtype: int64

Percentage:
urgency
medium    53.46
high      37.24
low        9.30
Name: proportion, dtype: float64

Classes:
0 -> high
1 -> low
2 -> medium


DATA SPLIT
Training: 77985
Validation: 9748
Testing: 9749


CLASS WEIGHTS
high -> 0.8950213469219116
low -> 3.586011863705339
medium -> 0.623500911445841


CREATING TEXT VECTORIZER
Vocabulary size: 367


BUILDING BiLSTM MODEL


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ ?                      │   0 (unbuilt) │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)



STARTING TRAINING
Epoch 1/15


c:\Users\Dell\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


610/610 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9859 - loss: 0.0293
Epoch 1: val_loss improved from None to 0.00013, saving model to sevasetu_urgency_bilstm.keras

Epoch 1: finished saving model to sevasetu_urgency_bilstm.keras
610/610 ━━━━━━━━━━━━━━━━━━━━ 936s 2s/step - accuracy: 0.9859 - loss: 0.0293 - val_accuracy: 1.0000 - val_loss: 1.3262e-04 - learning_rate: 0.0010
Epoch 2/15
610/610 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 1.0000 - loss: 7.4990e-05
Epoch 2: val_loss improved from 0.00013 to 0.00001, saving model to sevasetu_urgency_bilstm.keras

Epoch 2: finished saving model to sevasetu_urgency_bilstm.keras
610/610 ━━━━━━━━━━━━━━━━━━━━ 870s 1s/step - accuracy: 1.0000 - loss: 7.4990e-05 - val_accuracy: 1.0000 - val_loss: 9.9751e-06 - learning_rate: 0.0010
Epoch 3/15
610/610 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 1.0000 - loss: 2.7869e-05
Epoch 3: val_loss improved from 0.00001 to 0.00000, saving model to sevasetu_urgency_bilstm.keras

Epoch 3: finished saving model

ValueError: Invalid dtype: str4224

In [9]:
def predict_urgency(complaint):

    # Make sure input is a normal Python string
    complaint = str(complaint)

    # IMPORTANT:
    # Use a TensorFlow string tensor instead of NumPy string array
    input_tensor = tf.constant(
        [complaint],
        dtype=tf.string
    )

    # Predict
    probabilities = model.predict(
        input_tensor,
        verbose=0
    )[0]

    # Get predicted class
    predicted_index = int(
        np.argmax(probabilities)
    )

    # Convert class index -> urgency
    urgency = label_encoder.inverse_transform(
        [predicted_index]
    )[0]

    # Confidence
    confidence = float(
        probabilities[predicted_index]
    )

    # All probabilities
    probability_dict = {
        label_encoder.inverse_transform([i])[0]:
        float(probabilities[i])
        for i in range(len(probabilities))
    }

    return {
        "urgency": urgency,
        "confidence": confidence,
        "probabilities": probability_dict
    }
    
complaint ="mere marne ka mann ho raha hai."

result = predict_urgency(complaint)

print("Urgency:", result["urgency"])
print("Confidence:", f"{result['confidence'] * 100:.2f}%")

print("\nProbabilities:")

for label, probability in result["probabilities"].items():
    print(
        f"{label}: {probability * 100:.2f}%"
    )
    

Urgency: high
Confidence: 99.96%

Probabilities:
high: 99.96%
low: 0.04%
medium: 0.00%
